In [1]:
import json
import os
from tqdm import tqdm
from alias_cluster_ct6_no_zeroshot import process_get_final_answer
from collections import defaultdict
from sentence_transformers import SentenceTransformer


c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.4.1) was trained with spaCy v3.4.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)
def load_embedding_cache(path):
   
    with open(path, "w", encoding="utf-8") as f:
        json.dump({}, f)
    return {}

In [3]:
METADATA_PATH = r"data\metadata_medhop.json"
metadata = load_json(METADATA_PATH)
mapping_data = load_json(r'data\mapping_answer_gol_add_seq.json')

embedding_model = SentenceTransformer("pritamdeka/S-BioBert-snli-multinli-stsb",device="cpu")
CACHE_PATH = 'data/cache/embed_candidate_ans_cluster.json'
embed_candidate_cache = load_embedding_cache(CACHE_PATH)

# DIRECT

In [13]:
BASE_1000 = "data/ALL_ROUND/1000/medhop_results_gg_direct/all_ans" # KHÔNG MAPPING FINAL ANSWER
# BASE_1000 = "D:/DSKTLab/27-02-2026-CODE/medhop_results_gg_direct_ct6_mapping/all_ans" # CÓ MAPPING FINAL ANSWER

# Khởi tạo bộ lưu trữ thống kê
stats_qidx_clusters = {} 
stats_candidate_dist = defaultdict(int)

In [14]:
for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    
    if item['type_q'] != 'direct':
        continue
    
    file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue
            
    cluster_genered_answer = tmp_data['1']['0']['0']['cluster_genered_answer']
    if 'target_answer_type' in item:
        target_answer_type = item['target_answer_type']
    else:
        target_answer_type = tmp_data['1']['0']['0']['target_answer_type']
    
    # Thực hiện clustering và lấy kết quả kèm stats
    final_answer = process_get_final_answer(
        target_answer_type, 
        cluster_genered_answer, 
        mapping_data, 
        embedding_model, 
        embed_candidate_cache
    )
    
    # --- TRÍCH XUẤT THỐNG KÊ ---
    # Lấy thông tin từ key 'stats' mà mình đã thêm vào trong logic
    stats = final_answer.get("stats", {"num_clusters": 0, "cluster_sizes": []})
    
    # 1. Lưu số lượng cluster cho từng câu hỏi
    stats_qidx_clusters[f"{qidx}|0"] = stats["num_clusters"]
    
    # 2. Cộng dồn số lượng cluster dựa trên kích cỡ (số alias/candidate)
    for size in stats["cluster_sizes"]:
        stats_candidate_dist[size] += 1

# --- IN KẾT QUẢ THỐNG KÊ ---
print("\n--- THỐNG KÊ ALIAS CLUSTERING ---")
print(f"Tổng số QIDX đã xử lý: {len(stats_qidx_clusters)}")

print("\nPhân phối quy mô Cluster:")
# Sắp xếp theo số lượng candidate từ ít đến nhiều
for n_cand in sorted(stats_candidate_dist.keys()):
    count = stats_candidate_dist[n_cand]
    print(f"Cluster có {n_cand} candidates: {count} clusters")

100%|██████████| 1000/1000 [00:12<00:00, 82.60it/s]


--- THỐNG KÊ ALIAS CLUSTERING ---
Tổng số QIDX đã xử lý: 704

Phân phối quy mô Cluster:
Cluster có 1 candidates: 182 clusters
Cluster có 2 candidates: 67 clusters
Cluster có 3 candidates: 70 clusters
Cluster có 4 candidates: 54 clusters
Cluster có 5 candidates: 54 clusters
Cluster có 6 candidates: 85 clusters
Cluster có 7 candidates: 349 clusters


In [15]:
stats_candidate_dist

defaultdict(int, {4: 54, 3: 70, 1: 182, 7: 349, 2: 67, 5: 54, 6: 85})

In [17]:
# stats_qidx_clusters

# SEQUENCE

In [18]:
BASE_SEQ = "SEQ_medhop_results/all_ans"

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
    if item.get('type_q') != 'sequence':
        continue
        
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    
    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        # print(f"File not found: {file_path}")
        continue
  
    # Duyệt qua từng step trong dữ liệu sequence
    # Cấu trúc: tmp_data['1']['0'] chứa các key là '0', '1', '2'... (các step)
    for step_idx, step_content in tmp_data['1']['0'].items():
        # Lấy dữ liệu cần thiết cho clustering từ mỗi step
        cluster_genered_answer = step_content.get('cluster_genered_answer')
        
        # Xác định target_answer_type (ưu tiên từ metadata, sau đó đến trong file)
        if 'target_answer_type' in item:
            target_answer_type = item['target_answer_type']
        else:
            target_answer_type = step_content.get('target_answer_type', 'wh') # mặc định wh nếu thiếu

        if not cluster_genered_answer:
            continue

        # Thực hiện clustering cho từng step
        final_answer = process_get_final_answer(
            target_answer_type, 
            cluster_genered_answer, 
            mapping_data, 
            embedding_model, 
            embed_candidate_cache
        )
        
        # --- TRÍCH XUẤT THỐNG KÊ ---
        stats = final_answer.get("stats", {"num_clusters": 0, "cluster_sizes": []})
        
        # 1. Lưu số lượng cluster cho từng step của QIDX
        # Format: "QIDX|Step" (ví dụ: "q_1|0", "q_1|1")
        stats_qidx_clusters[f"{qidx}|{step_idx}"] = stats["num_clusters"]
        
        # 2. Cộng dồn vào phân phối quy mô cluster chung
        for size in stats["cluster_sizes"]:
            stats_candidate_dist[size] += 1

# --- IN KẾT QUẢ TỔNG HỢP (Cả Direct + Sequence) ---
print("\n" + "="*40)
print("THỐNG KÊ TỔNG HỢP (DIRECT & SEQUENCE)")
print("="*40)
print(f"Tổng số đơn vị xử lý (QIDX + Steps): {len(stats_qidx_clusters)}")

print("\nPhân phối quy mô Cluster (Alias Distribution):")
for n_cand in sorted(stats_candidate_dist.keys()):
    count = stats_candidate_dist[n_cand]
    print(f"Cluster có {n_cand:2d} candidates: {count:5d} clusters")

100%|██████████| 1000/1000 [00:38<00:00, 25.90it/s]


THỐNG KÊ TỔNG HỢP (DIRECT & SEQUENCE)
Tổng số đơn vị xử lý (QIDX + Steps): 1200

Phân phối quy mô Cluster (Alias Distribution):
Cluster có  1 candidates:   605 clusters
Cluster có  2 candidates:   150 clusters
Cluster có  3 candidates:   130 clusters
Cluster có  4 candidates:   103 clusters
Cluster có  5 candidates:   101 clusters
Cluster có  6 candidates:   143 clusters
Cluster có  7 candidates:   576 clusters


# PARALLEL

In [19]:
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"

for qidx in tqdm(list(metadata.keys())):    
    item = metadata[qidx]
       
    if item['type_q'] != 'parallel':
        continue
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    # Lặp qua các sub-questions (thường là 1, 2, 3, 4)
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break      
        
        # Lặp qua từng bước (step) của mỗi sub-question
        for step_idx, step_data in tmp_data[qi]['0'].items():
            step_str = str(step_idx)
            
            # Lấy dữ liệu thô để thực hiện clustering lại (hoặc lấy từ kết quả có sẵn)
            cluster_raw = step_data.get('cluster_genered_answer')
            if not cluster_raw:
                continue

            # Xác định type để chọn hàm YN hoặc WH
            if 'target_answer_type' in item:
                target_answer_type = item['target_answer_type']
            else:
                target_answer_type = step_data.get('target_answer_type', 'wh')

            # Thực hiện clustering
            final_answer = process_get_final_answer(
                target_answer_type, 
                cluster_raw, 
                mapping_data, 
                embedding_model, 
                embed_candidate_cache
            )
            
            # --- TRÍCH XUẤT THỐNG KÊ ---
            stats = final_answer.get("stats", {"num_clusters": 0, "cluster_sizes": []})
            
            # 1. Định danh duy nhất cho mỗi step trong parallel: qidx | sub-q | step
            unique_key = f"{qidx}|{qi}|{step_str}"
            stats_qidx_clusters[unique_key] = stats["num_clusters"]
            
            # 2. Cập nhật phân phối số lượng alias (candidate)
            for size in stats["cluster_sizes"]:
                stats_candidate_dist[size] += 1

# --- TỔNG KẾT TOÀN BỘ CÁC LOẠI ---
print("\n" + "="*50)
print("BÁO CÁO THỐNG KÊ CLUSTERING TỔNG HỢP")
print("="*50)
print(f"Tổng số mẫu (steps/questions) đã xử lý: {len(stats_qidx_clusters)}")

# Tính toán một vài chỉ số thông minh
total_clusters = sum(stats_candidate_dist.values())
avg_candidates = sum(k * v for k, v in stats_candidate_dist.items()) / total_clusters if total_clusters > 0 else 0

print(f"Tổng số cluster được tạo ra: {total_clusters}")
print(f"Số lượng candidate trung bình mỗi cluster: {avg_candidates:.2f}")
print("-" * 30)
print("Chi tiết phân phối Alias:")
for n_cand in sorted(stats_candidate_dist.keys()):
    count = stats_candidate_dist[n_cand]
    percentage = (count / total_clusters) * 100 if total_clusters > 0 else 0
    print(f"- Cluster có {n_cand:2d} candidates: {count:5d} cụm ({percentage:5.1f}%)")

100%|██████████| 1000/1000 [00:08<00:00, 121.76it/s]


BÁO CÁO THỐNG KÊ CLUSTERING TỔNG HỢP
Tổng số mẫu (steps/questions) đã xử lý: 1362
Tổng số cluster được tạo ra: 2118
Số lượng candidate trung bình mỗi cluster: 3.80
------------------------------
Chi tiết phân phối Alias:
- Cluster có  1 candidates:   760 cụm ( 35.9%)
- Cluster có  2 candidates:   176 cụm (  8.3%)
- Cluster có  3 candidates:   145 cụm (  6.8%)
- Cluster có  4 candidates:   123 cụm (  5.8%)
- Cluster có  5 candidates:   112 cụm (  5.3%)
- Cluster có  6 candidates:   157 cụm (  7.4%)
- Cluster có  7 candidates:   645 cụm ( 30.5%)


In [20]:
def run_clustering_stats(qidx, target_type, cluster_raw, step_key, stats_qidx, stats_dist):
    """
    Hàm dùng chung để gọi xử lý clustering và cập nhật vào dict thống kê.
    step_key: String dùng để định danh (ví dụ: qidx|step hoặc qidx|qi|step)
    """
    if not cluster_raw:
        return
        
    # Gọi hàm xử lý logic đã viết ở các bước trước
    final_res = process_get_final_answer(
        target_type, 
        cluster_raw, 
        mapping_data, 
        embedding_model, 
        embed_candidate_cache
    )
    
    # Lấy thông tin stats
    stats = final_res.get("stats", {"num_clusters": 0, "cluster_sizes": []})
    
    # Cập nhật Dict 1: {QIDX|... : số cluster}
    stats_qidx[step_key] = stats["num_clusters"]
    
    # Cập nhật Dict 2: {số alias : số lượng cluster}
    for size in stats["cluster_sizes"]:
        stats_dist[size] += 1

In [21]:
# Khởi tạo bộ lưu trữ dùng chung cho toàn bộ dự án
stats_qidx_clusters = {} 
stats_candidate_dist = defaultdict(int)

for qidx in tqdm(list(metadata.keys())):
    item = metadata[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file dựa trên loại câu hỏi
    if t_q == 'direct':
        base_path = BASE_1000
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        base_path = BASE_SEQ
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        base_path = BASE_PAR
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    # Xác định Target Answer Type (Ưu tiên metadata)
    ans_type = item.get('target_answer_type')

    # --- LOGIC BÓC TÁCH THEO TỪNG LOẠI ---
    
    if t_q == 'direct':
        # Direct: chỉ có 1 node cố định [1][0][0]
        step_data = tmp_data['1']['0']['0']
        curr_type = ans_type or step_data.get('target_answer_type', 'wh')
        run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                             f"{qidx}|1|0|0", stats_qidx_clusters, stats_candidate_dist)

    elif t_q == 'sequence':
        # Sequence: duyệt qua các step trong [1][0]
        for step_idx, step_data in tmp_data['1']['0'].items():
            curr_type = ans_type or step_data.get('target_answer_type', 'wh')
            run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                                 f"{qidx}|1|0|{step_idx}", stats_qidx_clusters, stats_candidate_dist)

    elif t_q == 'parallel':
        # Parallel: duyệt qua qi ["1","2","3","4"] rồi đến step
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: break
            for step_idx, step_data in tmp_data[qi]['0'].items():
                curr_type = ans_type or step_data.get('target_answer_type', 'wh')
                run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                                     f"{qidx}|{qi}|0|{step_idx}", stats_qidx_clusters, stats_candidate_dist)

# --- XUẤT BÁO CÁO CUỐI CÙNG ---
print("\n" + "="*50)
print(f"THỐNG KÊ TỔNG HỢP ({len(stats_qidx_clusters)} ĐƠN VỊ XỬ LÝ)")
print("="*50)
for size in sorted(stats_candidate_dist.keys()):
    print(f"Cụm có {size:2d} alias: {stats_candidate_dist[size]:5d} clusters")

100%|██████████| 1000/1000 [00:27<00:00, 36.08it/s]


THỐNG KÊ TỔNG HỢP (1362 ĐƠN VỊ XỬ LÝ)
Cụm có  1 alias:   760 clusters
Cụm có  2 alias:   176 clusters
Cụm có  3 alias:   145 clusters
Cụm có  4 alias:   123 clusters
Cụm có  5 alias:   112 clusters
Cụm có  6 alias:   157 clusters
Cụm có  7 alias:   645 clusters


In [25]:
BASE_9000 = "DIRECT_9000_medhop_results/all_ans"
BASE_SEQ = "SEQUENCE_9000_medhop_results/all_ans"
BASE_PAR = "PARALLEL_9000_medhop_results/all_ans"
metadata_9000 = load_json(r'data\metadata_9000q.json')

In [27]:
for qidx in tqdm(list(metadata_9000.keys())):
    item = metadata_9000[qidx]
    t_q = item.get('type_q')
    
    # Xác định đường dẫn file dựa trên loại câu hỏi
    if t_q == 'direct':
        base_path = BASE_9000
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'sequence':
        base_path = BASE_SEQ
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    elif t_q == 'parallel':
        base_path = BASE_PAR
        file_path = f"{base_path}/R4_step_3b_all_ans_{qidx}.json"
    else:
        continue

    try:
        tmp_data = load_json(file_path)
    except FileNotFoundError:
        continue

    # Xác định Target Answer Type (Ưu tiên metadata)
    ans_type = item.get('target_answer_type')

    # --- LOGIC BÓC TÁCH THEO TỪNG LOẠI ---
    
    if t_q == 'direct':
        # Direct: chỉ có 1 node cố định [1][0][0]
        step_data = tmp_data['1']['0']['0']
        curr_type = ans_type or step_data.get('target_answer_type', 'wh')
        run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                             f"{qidx}|1|0|0", stats_qidx_clusters, stats_candidate_dist)

    elif t_q == 'sequence':
        # Sequence: duyệt qua các step trong [1][0]
        for step_idx, step_data in tmp_data['1']['0'].items():
            curr_type = ans_type or step_data.get('target_answer_type', 'wh')
            run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                                 f"{qidx}|1|0|{step_idx}", stats_qidx_clusters, stats_candidate_dist)

    elif t_q == 'parallel':
        # Parallel: duyệt qua qi ["1","2","3","4"] rồi đến step
        for qi in ["1", "2", "3", "4"]:
            if qi not in tmp_data: break
            for step_idx, step_data in tmp_data[qi]['0'].items():
                curr_type = ans_type or step_data.get('target_answer_type', 'wh')
                run_clustering_stats(qidx, curr_type, step_data.get('cluster_genered_answer'), 
                                     f"{qidx}|{qi}|0|{step_idx}", stats_qidx_clusters, stats_candidate_dist)

# --- XUẤT BÁO CÁO CUỐI CÙNG ---
print("\n" + "="*50)
print(f"THỐNG KÊ TỔNG HỢP ({len(stats_qidx_clusters)} ĐƠN VỊ XỬ LÝ)")
print("="*50)
for size in sorted(stats_candidate_dist.keys()):
    print(f"Cụm có {size:2d} alias: {stats_candidate_dist[size]:5d} clusters")

100%|██████████| 9000/9000 [25:50<00:00,  5.80it/s]  


THỐNG KÊ TỔNG HỢP (12252 ĐƠN VỊ XỬ LÝ)
Cụm có  1 alias: 11212 clusters
Cụm có  2 alias:  2885 clusters
Cụm có  3 alias:  1752 clusters
Cụm có  4 alias:  1439 clusters
Cụm có  5 alias:  1286 clusters
Cụm có  6 alias:  1612 clusters
Cụm có  7 alias:  4216 clusters


In [1]:
save_json(stats_qidx_clusters, r'data\EVIDENCE\analysis\stats_qidx_clusters_row20.json')
save_json(stats_candidate_dist, r'data\EVIDENCE\analysis\stats_candidate_dist_row20.json')

NameError: name 'save_json' is not defined